# S23DR 2026 — HSS Evaluation

Evaluates a trained `RoofWireframeNet` checkpoint on the validation split
using the **Hausdorff Segment Score (HSS)**.

**Steps**
1. Check GPU
2. Mount Google Drive
3. Install dependencies & pull latest repo code
4. Load checkpoint from Drive
5. Load the S23DR validation dataset
6. Run inference + compute HSS
7. Results summary & confidence threshold sweep

## 1 · Check GPU

In [ ]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}  ({props.total_memory / 1e9:.1f} GB)")
    DEVICE = "cuda"
else:
    print("GPU      : not available — using CPU (will be slow)")
    DEVICE = "cpu"

print(f"Device   : {DEVICE}")

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CKPT_PATH = "/content/drive/MyDrive/s23dr_last.pt"   # ← change if needed
assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
print(f"Checkpoint found: {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)")

## 3 · Install dependencies & pull latest code

In [ ]:
!pip install -q datasets huggingface_hub scipy numpy

In [ ]:
import os

REPO_DIR = "/content/3d_building_construction"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/12turtleships/3d_building_construction {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 4 · Load checkpoint

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
from s23dr.model import RoofWireframeNet, WireframeLoss

device = torch.device(DEVICE)

try:
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location=device)

saved_args = ckpt.get("args", {})
N_QUERIES  = saved_args.get("n_queries", 64)
epoch      = ckpt.get("epoch", "?")
val_loss   = ckpt.get("val_loss", float("nan"))

print(f"epoch     = {epoch}")
print(f"val_loss  = {val_loss:.4f}")
print(f"n_queries = {N_QUERIES}")

model = RoofWireframeNet(n_queries=N_QUERIES).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print("Model loaded successfully.")

## 5 · Load the S23DR validation dataset

Streams from the public HF dataset — no local download needed.  
If you have the dataset saved locally via `save_to_disk`, set `USE_LOCAL = True`.

In [ ]:
# ── configuration ────────────────────────────────────────────────────────────
USE_LOCAL  = False                              # True → load from LOCAL_DATA_DIR
LOCAL_DATA_DIR = "/content/3d_building_construction/s23dr/data"
HF_DATASET = "usm3d/s23dr-2026-sampled_4096_v2"
SPLIT      = "validation"
N_POINTS   = 1024
BATCH_SIZE = 8
CONF_THRESH = 0.5   # vertex confidence threshold (tuned in step 7)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import io, zipfile
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


def _unpack_row(row):
    blob = row["data"]
    out = {}
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        for name in zf.namelist():
            if name.endswith(".npy"):
                out[name[:-4]] = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
    out["order_id"] = row.get("order_id", "")
    return out


class S23DRValDataset(Dataset):
    def __init__(self, rows, n_points=1024):
        self._rows = [_unpack_row(r) for r in rows]
        self.n_points = n_points

    def __len__(self):
        return len(self._rows)

    def __getitem__(self, idx):
        r = self._rows[idx]
        N   = min(self.n_points, len(r["xyz_norm"]))
        sel = np.arange(N)
        return {
            "order_id": r["order_id"],
            "xyz":       torch.from_numpy(r["xyz_norm"][sel]).float(),
            "vote_frac": torch.from_numpy(r["vote_frac"][sel]).float(),
            "n_views":   torch.from_numpy(r["n_views_voted"][sel].astype(np.float32)).float(),
            "mask":      torch.from_numpy(r["mask"][sel].astype(np.float32)).float(),
            "class_id":  torch.from_numpy(r["class_id"][sel].astype(np.int64)),
            "gt_segs":   torch.from_numpy(r["gt_segments"]).float(),  # (E, 2, 3)
        }


def _collate(batch):
    return {
        "order_id":  [b["order_id"]  for b in batch],
        "xyz":       torch.stack([b["xyz"]       for b in batch]),
        "vote_frac": torch.stack([b["vote_frac"] for b in batch]),
        "n_views":   torch.stack([b["n_views"]   for b in batch]),
        "mask":      torch.stack([b["mask"]      for b in batch]),
        "class_id":  torch.stack([b["class_id"]  for b in batch]),
        "gt_segs":   [b["gt_segs"].numpy()        for b in batch],
    }


# Load rows
if USE_LOCAL:
    from datasets import load_from_disk
    hf_ds = load_from_disk(LOCAL_DATA_DIR)
    rows  = list(hf_ds[SPLIT] if SPLIT in hf_ds else hf_ds)
else:
    from datasets import load_dataset
    rows = list(load_dataset(HF_DATASET, split=SPLIT))

val_ds = S23DRValDataset(rows, n_points=N_POINTS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, collate_fn=_collate)

print(f"Loaded {len(val_ds)} validation samples  (batch_size={BATCH_SIZE})")

In [ ]:
# ── Coordinate space diagnostic ─────────────────────────────────────────────
# Prints min/max/mean of each array to confirm pred_pos and gt_segments
# live in the same coordinate space.  If they differ by orders of magnitude
# the model will never score >0 HSS regardless of training quality.

r0 = val_ds._rows[0]   # raw unpacked dict for sample 0

print("── Raw dataset arrays (sample 0) ──")
for key in ("xyz_norm", "gt_vertices", "gt_segments"):
    arr = r0.get(key)
    if arr is None:
        print(f"  {key}: NOT FOUND")
    else:
        print(f"  {key}: shape={arr.shape}  "
              f"min={arr.min():.4f}  max={arr.max():.4f}  mean={arr.mean():.4f}")

# Run a single forward pass to inspect pred_pos range
_batch0 = val_ds[0]
with torch.no_grad():
    _out0 = model(
        _batch0["xyz"].unsqueeze(0).to(device),
        _batch0["vote_frac"].unsqueeze(0).to(device),
        _batch0["n_views"].unsqueeze(0).to(device),
        _batch0["mask"].unsqueeze(0).to(device),
        _batch0["class_id"].unsqueeze(0).to(device),
    )
pred_pos0 = _out0["pred_pos"][0].cpu().numpy()
print(f"\n  pred_pos (model output): shape={pred_pos0.shape}  "
      f"min={pred_pos0.min():.4f}  max={pred_pos0.max():.4f}  mean={pred_pos0.mean():.4f}")

# Also check if scale/center metadata is available
for key in ("scale", "center", "xyz_center", "xyz_scale"):
    val = r0.get(key)
    if val is not None:
        print(f"\n  {key}: {val}")

print("\n── Interpretation ──")
print("  gt_segments and pred_pos should be in the same range.")
print("  If gt_segments range is >> 1, it is in world space (metres)")
print("  and pred_pos (normalized) will never match → HSS = 0.")
print("  Fix: convert gt_segments to normalized space using scale/center,")
print("  OR convert pred_pos to world space before calling hss().")

## 6 · Diagnose model output, then run inference & compute HSS

In [ ]:
# Sanity-check a single batch before running the full eval.
# If edge_no_edge_frac ≈ 1.0 the model collapsed due to class imbalance
# during training → retrain with the fixed WireframeLoss (no_edge_weight=0.05).

diag_batch = next(iter(val_loader))
with torch.no_grad():
    diag_out = model(
        diag_batch["xyz"].to(device),
        diag_batch["vote_frac"].to(device),
        diag_batch["n_views"].to(device),
        diag_batch["mask"].to(device),
        diag_batch["class_id"].to(device),
    )

# Confidence distribution
confs = torch.sigmoid(diag_out["pred_conf"][0]).cpu()
print("── Vertex confidence (first sample) ──")
print(f"  min={confs.min():.3f}  max={confs.max():.3f}  mean={confs.mean():.3f}")
print(f"  active (>0.5): {(confs > 0.5).sum().item()}  "
      f"active (>0.1): {(confs > 0.1).sum().item()}")

# Edge prediction distribution
edge_cls = diag_out["edge_logits"][0].argmax(dim=-1).cpu()  # (K, K)
no_edge_class = 10
frac_no_edge = (edge_cls == no_edge_class).float().mean().item()
print("\n── Edge predictions (first sample) ──")
print(f"  Fraction predicting 'no edge' (class {no_edge_class}): {frac_no_edge:.4f}")
print(f"  Unique predicted classes: {edge_cls.unique().tolist()}")

if frac_no_edge > 0.999:
    print("\n⚠ Model predicts 'no edge' for every pair.")
    print("  Cause: class imbalance during training (no_edge_weight was missing).")
    print("  Fix:   retrain using the updated WireframeLoss(no_edge_weight=0.05).")
else:
    print("\n✓ Model is predicting some edges — proceeding to full eval.")

In [ ]:
import time
from s23dr.metrics import hss, decode_to_segments

loss_fn = WireframeLoss()


@torch.no_grad()
def evaluate(model, loader, conf_thresh):
    model.eval()
    results = []
    t0 = time.time()

    for i, batch in enumerate(loader):
        out = model(
            batch["xyz"].to(device),
            batch["vote_frac"].to(device),
            batch["n_views"].to(device),
            batch["mask"].to(device),
            batch["class_id"].to(device),
        )

        for b in range(batch["xyz"].shape[0]):
            verts, edges, _ = loss_fn.decode(
                out["pred_pos"][b],
                out["pred_conf"][b],
                out["edge_logits"][b],
                conf_thresh=conf_thresh,
            )
            pred_segs = decode_to_segments(verts, edges)
            gt_segs   = batch["gt_segs"][b]
            scores    = hss(pred_segs, gt_segs)
            results.append({"order_id": batch["order_id"][b], **scores})

        if (i + 1) % 10 == 0:
            mean_hss = np.mean([r["hss"] for r in results])
            print(f"  [{len(results):4d}/{len(loader.dataset)}]  "
                  f"mean_hss={mean_hss:.4f}  ({time.time()-t0:.1f}s)")

    return results


print(f"Evaluating with conf_thresh={CONF_THRESH} …")
eval_results = evaluate(model, val_loader, CONF_THRESH)
print(f"Done — {len(eval_results)} samples evaluated")

## 7 · Results summary & confidence threshold sweep

In [ ]:
import numpy as np

hss_scores  = np.array([r["hss"]       for r in eval_results])
prec_scores = np.array([r["precision"] for r in eval_results])
rec_scores  = np.array([r["recall"]    for r in eval_results])

print("=" * 45)
print(f"  conf_thresh : {CONF_THRESH}")
print(f"  Samples     : {len(hss_scores)}")
print("-" * 45)
print(f"  HSS         : {hss_scores.mean():.4f}  (std {hss_scores.std():.4f})")
print(f"  Precision   : {prec_scores.mean():.4f}")
print(f"  Recall      : {rec_scores.mean():.4f}")
print("=" * 45)

In [ ]:
# Sweep confidence thresholds to find the best HSS
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
sweep_results = []

print(f"{'thresh':>8}  {'HSS':>8}  {'Prec':>8}  {'Recall':>8}")
print("-" * 40)

for thresh in thresholds:
    res = evaluate(model, val_loader, conf_thresh=thresh)
    h = np.mean([r["hss"]       for r in res])
    p = np.mean([r["precision"] for r in res])
    rc = np.mean([r["recall"]   for r in res])
    sweep_results.append((thresh, h, p, rc))
    print(f"{thresh:>8.2f}  {h:>8.4f}  {p:>8.4f}  {rc:>8.4f}")

best = max(sweep_results, key=lambda x: x[1])
print("-" * 40)
print(f"Best conf_thresh = {best[0]}  →  HSS = {best[1]:.4f}")

In [ ]:
# Inspect the 10 worst-scoring samples for error analysis
sorted_results = sorted(eval_results, key=lambda r: r["hss"])

print("10 worst samples:")
print(f"{'order_id':>30}  {'HSS':>8}  {'Prec':>8}  {'Recall':>8}")
print("-" * 60)
for r in sorted_results[:10]:
    print(f"{str(r['order_id']):>30}  {r['hss']:>8.4f}  "
          f"{r['precision']:>8.4f}  {r['recall']:>8.4f}")

In [ ]:
# Optional: save per-sample results to Drive
import json, os

out_path = "/content/drive/MyDrive/s23dr_eval_results.json"
with open(out_path, "w") as f:
    json.dump({
        "checkpoint": CKPT_PATH,
        "split": SPLIT,
        "conf_thresh": CONF_THRESH,
        "mean_hss":       float(hss_scores.mean()),
        "mean_precision": float(prec_scores.mean()),
        "mean_recall":    float(rec_scores.mean()),
        "per_sample": eval_results,
    }, f, indent=2)

print(f"Results saved to {out_path}")

In [ ]:
# ── 8 · Visualise point cloud + predicted vs GT wireframe ────────────────────
# Interactive 3-D plotly chart for a single validation sample.
#
# Blue dots   = input point cloud (xyz_norm)
# Green lines = GT roof wireframe (gt_segments, normalised space)
# Red lines   = Predicted roof wireframe (at VIS_THRESH)
# Red diamonds= Predicted vertices

import plotly.graph_objects as go

SAMPLE_IDX = 0      # ← change to inspect a different sample
VIS_THRESH = 0.10   # confidence threshold (best from the sweep above)
MAX_PTS    = 2048   # subsample point cloud for faster rendering

# ── raw data ──────────────────────────────────────────────────────────────────
r        = val_ds._rows[SAMPLE_IDX]
xyz_np   = r["xyz_norm"]        # (N, 3)
gt_segs  = r["gt_segments"]     # (E, 2, 3)  normalised space

# ── inference ─────────────────────────────────────────────────────────────────
item = val_ds[SAMPLE_IDX]
with torch.no_grad():
    out = model(
        item["xyz"].unsqueeze(0).to(device),
        item["vote_frac"].unsqueeze(0).to(device),
        item["n_views"].unsqueeze(0).to(device),
        item["mask"].unsqueeze(0).to(device),
        item["class_id"].unsqueeze(0).to(device),
    )
verts, edges, _ = loss_fn.decode(
    out["pred_pos"][0], out["pred_conf"][0], out["edge_logits"][0],
    conf_thresh=VIS_THRESH,
)
pred_segs = decode_to_segments(verts, edges)   # (E_pred, 2, 3)

# ── point cloud trace ─────────────────────────────────────────────────────────
rng = np.random.default_rng(0)
vis_idx = rng.choice(len(xyz_np), min(MAX_PTS, len(xyz_np)), replace=False)
pc = xyz_np[vis_idx]

pc_trace = go.Scatter3d(
    x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
    mode="markers",
    marker=dict(size=1.5, color="royalblue", opacity=0.35),
    name="Point cloud",
)

# ── helper: (E, 2, 3) → NaN-separated line trace ─────────────────────────────
def _line_trace(segs, color, name, width=4):
    if len(segs) == 0:
        return go.Scatter3d(x=[], y=[], z=[], mode="lines",
                            line=dict(color=color, width=width), name=name)
    xs, ys, zs = [], [], []
    for s in segs:
        xs += [float(s[0, 0]), float(s[1, 0]), None]
        ys += [float(s[0, 1]), float(s[1, 1]), None]
        zs += [float(s[0, 2]), float(s[1, 2]), None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                        line=dict(color=color, width=width), name=name)

gt_trace   = _line_trace(gt_segs,   color="limegreen", name=f"GT ({len(gt_segs)} segs)")
pred_trace = _line_trace(pred_segs, color="red",        name=f"Pred ({len(pred_segs)} segs)")

# ── predicted vertex markers ──────────────────────────────────────────────────
verts_np = verts.cpu().numpy() if hasattr(verts, "cpu") else np.array(verts)
vert_trace = go.Scatter3d(
    x=verts_np[:, 0] if len(verts_np) else [],
    y=verts_np[:, 1] if len(verts_np) else [],
    z=verts_np[:, 2] if len(verts_np) else [],
    mode="markers",
    marker=dict(size=6, color="red", symbol="diamond"),
    name=f"Pred vertices ({len(verts_np)})",
)

hss_score = hss(pred_segs, gt_segs)

fig = go.Figure(data=[pc_trace, gt_trace, pred_trace, vert_trace])
fig.update_layout(
    title=(f"Sample {SAMPLE_IDX} | {r['order_id']} | thresh={VIS_THRESH} | "
           f"HSS={hss_score['hss']:.3f}  P={hss_score['precision']:.3f}  R={hss_score['recall']:.3f}"),
    scene=dict(aspectmode="data",
               xaxis_title="X (norm)", yaxis_title="Y (norm)", zaxis_title="Z (norm)"),
    legend=dict(x=0, y=1),
    margin=dict(l=0, r=0, t=50, b=0),
    height=680,
)
fig.show()

print(f"GT segments   : {len(gt_segs)}")
print(f"Pred segments : {len(pred_segs)}")
print(f"Pred vertices : {len(verts_np)}")
print(f"HSS={hss_score['hss']:.4f}  Precision={hss_score['precision']:.4f}  Recall={hss_score['recall']:.4f}")

## 9 · xyz_norm Line-Structure Analysis

**Why does the point cloud appear to have so many lines?**

Three hypotheses were ruled out by diagnostics:

| Hypothesis | Test | Result | Verdict |
|---|---|---|---|
| `mode='lines'` in plot | Check plot code | `mode='markers'` confirmed | ✗ ruled out |
| NaN line-list format | `np.isnan(xyz_norm).any()` | `False` | ✗ ruled out |
| Surface normals (unit vectors) | `np.linalg.norm(…)` p50 | `0.33` (not ≈ 1.0) | ✗ ruled out |
| Outlier axis compression | 134/4096 points norm>0.7 | Axes balanced (X:1.36, Y:0.65, Z:1.6) | Minor contributor |

**Confirmed root cause:** The lines are genuine architectural edge features recorded by the
SfM reconstruction. Window-frame edges, building corners, and wall junctions generate dense
clusters of collinear points (dozens spaced millimetres apart on the same vertical edge).
At `size=1.5` in Plotly these render as solid-looking streaks.

**Implication for training:** The point cloud is correct. The model must learn to detect
roof wireframe vertices and edges from a cloud that is dominated by facade edge structure.
The `mask` and `class_id` fields segment roof vs. facade points and are the key inputs
for suppressing this facade-line noise during inference.

**Visualisation tip:** Use `aspectmode='cube'` (equal visual scale for all axes) to
break the streaks into visible individual dots, as shown in the cell below.

In [ ]:
# ── xyz_norm line-structure visualisation ────────────────────────────────────
# Splits into inliers (norm ≤ 0.6, ~97%) and outliers (norm > 0.7, ~3%).
# aspectmode='cube' gives Z equal visual scale → streaks break into dots.

import numpy as np
import plotly.graph_objects as go

SAMPLE_IDX = 0
r = val_ds._rows[SAMPLE_IDX]
xyz_norm = r["xyz_norm"]

norms = np.linalg.norm(xyz_norm, axis=1)
print(f"shape                       : {xyz_norm.shape}")
print(f"X range                     : [{xyz_norm[:,0].min():.3f}, {xyz_norm[:,0].max():.3f}]")
print(f"Y range                     : [{xyz_norm[:,1].min():.3f}, {xyz_norm[:,1].max():.3f}]")
print(f"Z range                     : [{xyz_norm[:,2].min():.3f}, {xyz_norm[:,2].max():.3f}]")
print(f"Norm p50/p90/p99/max        : {np.percentile(norms, [50,90,99,100]).round(3)}")
print(f"Outliers norm > 0.7         : {(norms > 0.7).sum()} / {len(norms)}")

inlier_mask = norms <= 0.6
xyz_in  = xyz_norm[inlier_mask]
xyz_out = xyz_norm[~inlier_mask]
print(f"Inliers (norm ≤ 0.6)        : {inlier_mask.sum()}")

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=xyz_in[:,0], y=xyz_in[:,1], z=xyz_in[:,2],
    mode="markers",
    marker=dict(size=1.5, color="royalblue", opacity=0.5),
    name=f"Inliers norm≤0.6 ({inlier_mask.sum()})",
))
if (~inlier_mask).sum() > 0:
    fig.add_trace(go.Scatter3d(
        x=xyz_out[:,0], y=xyz_out[:,1], z=xyz_out[:,2],
        mode="markers",
        marker=dict(size=4, color="red", opacity=0.9),
        name=f"Outliers norm>0.7 ({(~inlier_mask).sum()})",
    ))
fig.update_layout(
    title=(
        f"xyz_norm line analysis — sample {SAMPLE_IDX} | {r['order_id']}  "
        "(aspectmode='cube' → equal Z scale, streaks become dots)"
    ),
    scene=dict(
        aspectmode="cube",
        xaxis_title="X (norm)", yaxis_title="Y (norm)", zaxis_title="Z (norm)",
    ),
    height=680, margin=dict(l=0, r=0, t=60, b=0),
)
fig.show()
print("Lines persist after filtering → real SfM architectural edges (building corners,")
print("window frames, wall junctions). Expected for urban facade datasets.")

## 10 · Note: API Error in long sessions

If you see:
```
API Error: 400 messages.N.content.1.text: cache_control cannot be set for empty text blocks
API Error: 400 messages: text content blocks must be non-empty
```

This is a **Claude Code on the web session infrastructure error**, not a bug in this notebook.
It occurs when the conversation history grows long (~20+ turns) and the system tries to apply
prompt caching (`cache_control`) to a content block that ended up empty.

**This notebook contains no Anthropic API calls** — the error is in the chat session wrapping
the notebook, not in the notebook code itself.

**Fix:** Start a fresh Claude Code session (the conversation context resets, clearing the
malformed block). The notebook code and data are unaffected.